# Notebook 2 – Logistic Regression for Sleep‑Stage Classification

In this notebook we build a **baseline classifier** that predicts the classic sleep stages (Wake, N1, N2, N3, REM) from **single‑channel EEG spectral features** computed in Notebook 1.  

**Objectives**
1. Load the pre‑computed feature table (`eeg_features.csv`).  
2. Prepare feature matrix **X** and label vector **y**.  
3. Build a `Pipeline` → `StandardScaler` + `LogisticRegression` (one‑vs‑rest).  
4. Evaluate with 5‑fold stratified cross‑validation (accuracy & F1).  
5. Visualise the confusion matrix and discuss results.

## 1. Imports and data loading

We first import the required libraries and load the EEG feature table generated in Notebook 1 (30‑second epochs × spectral power features).  
If the CSV is missing, run Notebook 1 to create it.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
FEATURE_PATH = Path("./outputs/sleep_edf_features.csv")
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        "Feature file 'sleep_edf_features.csv' not found. "
        "Run Notebook 1 first (or check the path)."
    )

df = pd.read_csv(FEATURE_PATH)
print(f"Loaded feature table with shape {df.shape}")
display(df.head())

## 2. Prepare `X` (features) and `y` (labels)

The feature columns correspond to relative band powers in the δ, θ, α, σ and β ranges.  
The `stage` column contains the target labels using R&K nomenclature (`W`, `N1`, `N2`, `N3`, `REM`).

In [ ]:
# --------------------------------------------------------------------
# 2 – Prepare X and y
# --------------------------------------------------------------------
# Accept both naming conventions:
bands = ["delta", "theta", "alpha", "sigma", "beta"]
if all(b in df.columns for b in bands):                     # no prefix
    feature_cols = bands
elif all(f"pow_{b}" in df.columns for b in bands):          # with 'pow_' prefix
    feature_cols = [f"pow_{b}" for b in bands]
else:
    raise ValueError("Band-power columns not found. Check Notebook 1 output.")

X = df[feature_cols].values

# Handle labels stored as strings **or** integers
if pd.api.types.is_numeric_dtype(df["stage"]):
    y = df["stage"].astype(int).values
else:
    string_to_int = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "REM": 4}
    y = df["stage"].map(string_to_int).astype(int).values

# ------------------------------------------------------------------
# Label names in numeric order – used later for the confusion matrix
# ------------------------------------------------------------------
LABEL_NAMES = ["W", "N1", "N2", "N3", "REM"]

print("Class distribution (y):")
print(pd.Series(y).value_counts().sort_index())

## 3. Build the Logistic Regression pipeline & cross‑validation

We standardise the features and fit a **one‑vs‑rest Logistic Regression**.  
Stratified 5‑fold CV keeps the class balance consistent across folds.

In [ ]:
# --------------------------------------------------------------------
# 3 – Build pipeline and evaluate with 5-fold stratified CV
# --------------------------------------------------------------------
pipe = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "logreg",
            OneVsRestClassifier(
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="liblinear",  # good for OvR
                )
            ),
        ),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    pipe,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=-1,
    return_estimator=True,
)

print(
    f"Mean accuracy: {cv_results['test_accuracy'].mean():.3f} ± "
    f"{cv_results['test_accuracy'].std():.3f}"
)
print(
    f"Mean macro-F1: {cv_results['test_f1_macro'].mean():.3f} ± "
    f"{cv_results['test_f1_macro'].std():.3f}"
)

## 4. Confusion matrix of the first fold

Let’s inspect how the classifier confuses specific stages by plotting the confusion matrix for the first CV split.

In [ ]:
# --------------------------------------------------------------------
# 4 – Confusion matrix (first CV fold) with row percentages
# --------------------------------------------------------------------

# Take the first split / estimator from cross-validation
train_idx, test_idx = list(cv.split(X, y))[0]
estimator0 = cv_results["estimator"][0]

y_pred = estimator0.predict(X[test_idx])

# Raw counts
cm = confusion_matrix(y[test_idx], y_pred, labels=range(len(LABEL_NAMES)))

# Row-normalised percentages (avoid div/0 with np.errstate)
with np.errstate(all="ignore"):
    cm_perc = cm / cm.sum(axis=1, keepdims=True) * 100
    cm_perc = np.nan_to_num(cm_perc)  # if a class is absent

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_perc,
    annot=cm,                    # numbers inside the boxes = counts
    fmt="d",
    cmap="Blues",
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    cbar_kws={"label": "% of true class"},
    linewidths=.5,
    linecolor="grey",
    ax=ax,
)

ax.set_xlabel("Predicted stage")
ax.set_ylabel("True stage")
ax.set_title("Confusion matrix – counts (annot) & row % (colour)")
plt.tight_layout()
plt.show()

## 5. Commentary

* The linear model captures most variance with surprisingly good accuracy on the dominant stages (N2, REM, Wake).  
* Transient stage **N1** remains the hardest to detect – typical given limited samples and its spectral overlap with Wake.  
* Balancing classes (`class_weight='balanced'`) mitigates bias but cannot fully solve overlap issues.  
* In future notebooks we’ll explore non‑linear models (e.g., Random Forest, Gradient Boosting) and multimodal fusion to improve minority‑class performance.